# Demo F: vLLM Engine Benchmarks

**Platform:** Lightning.ai Studio (A100 GPU)

**Goal:** Prove that vLLM with optimizations gives 10-20x improvement over HuggingFace.
Same model, same GPU, different flags.

| Experiment | Optimization | vLLM Flag | Expected Gain |
|---|---|---|---|
| 0 | KV Cache Visualization | (default) | See the problem |
| 1 | HF Baseline | (no engine) | 1x |
| 2 | PagedAttention + Batching | (vLLM default) | 5-10x throughput |
| 3 | Prefix Caching | `--enable-prefix-caching` | 5-15x TTFT |
| 4 | KV Quantization | `--kv-cache-dtype fp8` | 2x capacity |
| 5 | Speculative Decoding | `--speculative-model` | 2-3x decode |

## Setup on Lightning.ai:
1. Create a Studio with A100 GPU
2. Install: `pip install vllm openai`
3. Download model: `hf download mistralai/Mistral-7B-v0.1`
4. Start vLLM server in terminal (commands provided per experiment)
5. Run this notebook top to bottom


In [ ]:
# --- Setup: Install dependencies + define all utilities ---
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'openai', 'torch', 'transformers', 'accelerate',
    'matplotlib', 'requests', 'tqdm', 'numpy<2', 'scipy>=1.14'])

import torch, time, requests, re, threading
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
from transformers import AutoTokenizer, AutoModelForCausalLM

# ─── Config ───
MODEL = 'mistralai/Mistral-7B-v0.1'
PORT = 8000
VLLM_URL = f'http://localhost:{PORT}'
BASE_URL = f'{VLLM_URL}/v1'
WARMUP_PROMPT = 'This is a warmup call to compile CUDA kernels and warm memory.'
N_REQUESTS = 10
N_TOKENS = 50

# OpenAI-compatible client (works with vLLM + SGLang)
client = OpenAI(base_url=BASE_URL, api_key='unused')

# Standard test prompts (diverse to avoid caching bias)
PROMPTS = [
    'Explain the concept of attention mechanisms in transformers',
    'What are the key differences between CPU and GPU architectures',
    'Describe how PagedAttention works in vLLM',
    'What is the roofline model for GPU performance analysis',
    'Explain continuous batching in LLM serving systems',
    'How does speculative decoding accelerate inference',
    'What are the tradeoffs of KV cache quantization',
    'Describe the prefill vs decode phases of LLM inference',
    'How do mixture of experts models reduce compute costs',
    'What is tensor parallelism and when should you use it',
]

# Prefix prompts: shared system context + varying questions
SYSTEM_PREFIX = (
    'You are an expert systems architect specializing in distributed ML infrastructure. '
    'You have deep knowledge of GPU memory hierarchies, CUDA programming, network topologies, '
    'and large-scale model serving. Answer questions precisely and technically. '
    'Focus on practical production considerations over theoretical ideals. '
    'Always mention relevant tradeoffs and failure modes.'
)  # ~100 tokens shared prefix

PREFIX_QUESTIONS = [
    'How should I handle GPU OOM errors in production?',
    'What monitoring metrics matter for LLM serving?',
    'Compare disaggregated prefill vs colocated serving.',
    'How does request scheduling affect tail latency?',
    'What causes throughput degradation under high concurrency?',
]

# ─── Utilities ───
def check_server():
    """Check if vLLM server is running."""
    try:
        r = requests.get(f'{VLLM_URL}/health', timeout=3)
        if r.status_code == 200:
            models = requests.get(f'{BASE_URL}/models', timeout=3).json()
            print(f'vLLM server running. Model: {models["data"][0]["id"]}')
            return True
    except Exception:
        pass
    print('vLLM server NOT running. Start it in terminal first.')
    return False

def get_vllm_metrics():
    """Scrape vLLM /metrics endpoint, return dict of metric_name: value."""
    r = requests.get(f'{VLLM_URL}/metrics')
    metrics = {}
    for line in r.text.split('\n'):
        if line and not line.startswith('#'):
            parts = line.split()
            if len(parts) >= 2:
                try:
                    metrics[parts[0]] = float(parts[-1])
                except ValueError:
                    pass
    return metrics

def benchmark_concurrent(prompts, max_tokens=50, n_workers=10, label=''):
    """Send prompts concurrently via OpenAI SDK streaming. Returns metrics dict."""
    def single_request(prompt):
        t0 = time.perf_counter()
        first_token_time = None
        tokens = 0
        stream = client.completions.create(
            model=MODEL, prompt=prompt, max_tokens=max_tokens,
            temperature=0, stream=True
        )
        for chunk in stream:
            if first_token_time is None:
                first_token_time = time.perf_counter()
            tokens += 1
        elapsed = time.perf_counter() - t0
        ttft = (first_token_time - t0) if first_token_time else elapsed
        return {'ttft': ttft, 'tokens': tokens, 'elapsed': elapsed}

    t_wall = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        futures = [pool.submit(single_request, p) for p in prompts]
        req_results = [f.result() for f in tqdm(as_completed(futures), total=len(prompts), desc=label)]
    wall_time = time.perf_counter() - t_wall

    total_tokens = sum(r['tokens'] for r in req_results)
    avg_ttft = np.mean([r['ttft'] for r in req_results])
    throughput = total_tokens / wall_time

    print(f'  [{label}] {len(prompts)} reqs | {throughput:.0f} tok/s | TTFT: {avg_ttft*1000:.0f}ms | wall: {wall_time:.1f}s')
    return {'throughput_tps': throughput, 'avg_ttft': avg_ttft, 'wall_time': wall_time, 'total_tokens': total_tokens}

def plot_comparison(results, title='Running Comparison (vs HF Baseline)'):
    """Bar chart of all results collected so far."""
    valid = {k: v for k, v in results.items() if v is not None}
    names = list(valid.keys())
    tps = [valid[k]['throughput_tps'] for k in names]
    colors = ['#ffe4e6'] + ['#dcfce7'] * (len(names) - 1)

    fig, ax = plt.subplots(figsize=(max(6, len(names)*2), 4))
    ax.bar(names, tps, color=colors, edgecolor='#000', linewidth=1.2)
    for i, (n, tp) in enumerate(zip(names, tps)):
        ax.text(i, tp + max(tps)*0.02, f'{tp:.0f}', ha='center', fontsize=10, fontweight='bold')
        if i > 0:
            ax.text(i, tp*0.5, f'{tp/tps[0]:.1f}x', ha='center', fontsize=12, color='#166534', fontweight='bold')
    ax.set_ylabel('Throughput (tok/s)')
    ax.set_title(title, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

# Store results across all experiments
results = {}
print('Setup complete.')
print(f'  Model: {MODEL}')
print(f'  Server: {VLLM_URL}')
print(f'  Test: {N_REQUESTS} prompts x {N_TOKENS} tokens')


## Experiment 0: Observing KV Cache in Action

Before benchmarking, let's SEE the KV cache growing.
vLLM exposes `/metrics` (Prometheus format) showing KV cache utilization.

We send increasing concurrent requests with LONG outputs and sample peak KV usage.

**Start vLLM server first:**
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --gpu-memory-utilization 0.60 \
    --port 8000
```


In [ ]:
# --- Experiment 0: KV Cache Visualization ---
assert check_server(), 'Start vLLM server first!'

kv_usage_over_load = []
load_levels = [1, 5, 10, 20, 30]

print('Watching KV cache fill as we increase concurrent users...')
for n_conc in tqdm(load_levels, desc='Load levels'):
    conc_prompts = [f'Write a very detailed essay about topic {i}:' for i in range(n_conc)]
    peak_kv = 0.0

    def send_requests():
        with ThreadPoolExecutor(max_workers=n_conc) as executor:
            futures = [executor.submit(
                requests.post, f'{BASE_URL}/completions',
                json={'model': MODEL, 'prompt': p, 'max_tokens': 300, 'temperature': 0.7}
            ) for p in conc_prompts]
            [f.result() for f in futures]  # wait for all

    # Run requests in background, sample KV multiple times
    t = threading.Thread(target=send_requests)
    t.start()
    time.sleep(1.5)  # let prefill start
    for _ in range(5):
        m = get_vllm_metrics()
        kv = next((v for k, v in m.items() if 'kv_cache_usage_perc' in k), 0) * 100
        peak_kv = max(peak_kv, kv)
        time.sleep(0.5)
    t.join()

    kv_usage_over_load.append((n_conc, peak_kv))
    print(f'  {n_conc:>2} users -> peak KV: {peak_kv:.1f}%')

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
loads = [x[0] for x in kv_usage_over_load]
pcts = [x[1] for x in kv_usage_over_load]
ax.plot(loads, pcts, 'o-', color='#991b1b', linewidth=2, markersize=8)
ax.axhline(y=90, color='#64748b', linestyle='--', label='90% limit')
ax.set_xlabel('Concurrent Users')
ax.set_ylabel('KV Cache Usage (%)')
ax.set_title('KV Cache Fills Up With Concurrent Users', fontweight='bold')
ax.set_ylim(0, 100)
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for x, y in zip(loads, pcts):
    ax.annotate(f'{y:.0f}%', (x, y), textcoords='offset points', xytext=(0, 10), ha='center')
plt.tight_layout()
plt.show()
print('\nThis is the bottleneck. Every optimization reduces KV pressure or improves throughput.')


## Experiment 1: HuggingFace Baseline (Sequential)

No engine. No batching. No paging. Just `model.generate()` one request at a time.
This establishes the floor we're improving from.

**Stop vLLM server** (Ctrl+C) before running this cell (needs GPU memory).


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

# --- Experiment 1: HuggingFace Baseline ---
print('Loading Mistral-7B via HuggingFace...')
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL)
hf_tokenizer.pad_token_id = hf_tokenizer.eos_token_id
hf_model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map='auto')

# Warmup: 3 calls with DIFFERENT prompt to compile CUDA kernels
print('Warming up (3 calls)...')
for i in range(3):
    w_ids = hf_tokenizer(f'Warmup call number {i}', return_tensors='pt').input_ids.to('cuda')
    _ = hf_model.generate(w_ids, max_new_tokens=5, pad_token_id=hf_tokenizer.eos_token_id)
torch.cuda.synchronize()

# Benchmark: sequential generation
hf_ttfts = []
hf_total_tokens = 0
torch.cuda.synchronize()
hf_start = time.perf_counter()

for prompt in tqdm(PROMPTS, desc='HF baseline'):
    input_ids = hf_tokenizer(prompt, return_tensors='pt').input_ids.to('cuda')
    t0 = time.perf_counter()
    with torch.no_grad():
        output = hf_model.generate(input_ids, max_new_tokens=N_TOKENS, pad_token_id=hf_tokenizer.eos_token_id)
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    generated = output.shape[1] - input_ids.shape[1]
    hf_ttfts.append(t1 - t0)
    hf_total_tokens += generated

hf_wall = time.perf_counter() - hf_start
hf_throughput = hf_total_tokens / hf_wall

results['HF Baseline'] = {
    'throughput_tps': hf_throughput, 'avg_ttft': np.mean(hf_ttfts),
    'wall_time': hf_wall, 'total_tokens': hf_total_tokens
}
print(f'HF: {hf_throughput:.1f} tok/s, avg latency {np.mean(hf_ttfts)*1000:.0f}ms')

# Free GPU for vLLM
del hf_model, hf_tokenizer
torch.cuda.empty_cache()
import gc; gc.collect()
print('GPU freed. Start vLLM server now.')


In [ ]:
plot_comparison(results)

## Experiment 2: vLLM (PagedAttention + Continuous Batching)

Start vLLM server:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --gpu-memory-utilization 0.60 \
    --port 8000
```

Wait for `Uvicorn running on http://0.0.0.0:8000`.

Note: In vLLM 0.23+, prefix caching is enabled by default. This benchmark
uses diverse prompts (no shared prefix) so prefix caching has no effect here.


In [ ]:
# --- Experiment 2: vLLM Default ---
assert check_server(), 'Start vLLM server first!'

# Warmup: 5 requests to fully compile CUDA graphs
print('Warming up vLLM (5 requests)...')
for i in range(5):
    _ = requests.post(f'{BASE_URL}/completions',
        json={'model': MODEL, 'prompt': f'Warmup request {i} to compile CUDA graphs', 'max_tokens': 20})

# Concurrent benchmark with same prompts as HF
vllm_results = benchmark_concurrent(PROMPTS, max_tokens=N_TOKENS, n_workers=N_REQUESTS, label='vLLM default')
results['vLLM'] = vllm_results

print(f"\nSpeedup over HF: {vllm_results['throughput_tps']/results['HF Baseline']['throughput_tps']:.1f}x")


In [ ]:
plot_comparison(results)


## Experiment 3: Static Prefix Caching

Many apps share a long system prompt. Without prefix caching, vLLM recomputes
the KV cache for the shared prefix every time. With it, first request pays full
cost, subsequent requests skip the shared prefix computation.

Stop the server and restart with prefix caching:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --enable-prefix-caching \
    --gpu-memory-utilization 0.60 \
    --port 8000
```

We send requests SEQUENTIALLY with the same system prompt to clearly show
the first request (cold) vs subsequent requests (warm).


In [ ]:
# --- Experiment 3: Prefix Caching (Sequential) ---
assert check_server()

# Check prefix cache metrics before
before_metrics = get_vllm_metrics()

# Send requests SEQUENTIALLY with same system prefix
timings = []
print(f'Sending {len(PREFIX_QUESTIONS)} requests with SAME system prompt (~100 tokens)...\n')

for i, q in enumerate(PREFIX_QUESTIONS):
    prompt = f'{SYSTEM_PREFIX} {q}'
    t0 = time.perf_counter()
    # Use streaming to get TTFT
    first_token_time = None
    tokens = 0
    stream = client.completions.create(
        model=MODEL, prompt=prompt, max_tokens=N_TOKENS, temperature=0, stream=True
    )
    for chunk in stream:
        if first_token_time is None:
            first_token_time = time.perf_counter()
        tokens += 1
    elapsed = time.perf_counter() - t0
    ttft = (first_token_time - t0) if first_token_time else elapsed
    timings.append({'ttft': ttft, 'elapsed': elapsed, 'tokens': tokens})
    tag = 'COLD' if i == 0 else 'WARM'
    print(f'  [{tag}] {q[:40]:<40} TTFT: {ttft*1000:.0f}ms  total: {elapsed:.2f}s')

# Check metrics after
after_metrics = get_vllm_metrics()
prefix_queries_before = before_metrics.get('vllm:prefix_cache_queries_total', 0)
prefix_queries_after = after_metrics.get('vllm:prefix_cache_queries_total', 0)
print(f'\nPrefix cache queries: {prefix_queries_before:.0f} -> {prefix_queries_after:.0f} (+{prefix_queries_after-prefix_queries_before:.0f})')

# Compute results (use warm average)
cold_ttft = timings[0]['ttft']
warm_ttfts = [t['ttft'] for t in timings[1:]]
warm_avg_ttft = np.mean(warm_ttfts)
total_tokens = sum(t['tokens'] for t in timings)
total_wall = sum(t['elapsed'] for t in timings)

print(f'\nCold TTFT: {cold_ttft*1000:.0f}ms -> Warm TTFT: {warm_avg_ttft*1000:.0f}ms')
print(f'Prefix caching TTFT speedup: {cold_ttft/warm_avg_ttft:.2f}x')

# Concurrent benchmark for fair throughput comparison (same concurrency as Exp 2)
print('\nConcurrent benchmark (for throughput chart)...')
prefix_concurrent = benchmark_concurrent(PROMPTS, max_tokens=N_TOKENS, n_workers=N_REQUESTS, label='prefix concurrent')
prefix_concurrent['avg_ttft'] = warm_avg_ttft  # use the sequential warm TTFT (more meaningful)
results['vLLM+Prefix'] = prefix_concurrent


In [ ]:
plot_comparison(results)


## Experiment 4: Vllm + Prefix Caching + KV Cache Quantization

Stop the server and restart with FP8 KV cache:
```bash

python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --enable-prefix-caching \
    --kv-cache-dtype fp8 \
    --gpu-memory-utilization 0.60 \
    --port 8000

```

FP8 KV cache halves memory per token. Same concurrency test as Experiment 2
to show throughput is maintained, plus a stress test at 20 users.


In [ ]:
# --- Experiment 4: KV Quantization ---
assert check_server()

# Warmup: 5 requests to fully compile CUDA graphs
for i in range(5):
    _ = requests.post(f'{BASE_URL}/completions',
        json={'model': MODEL, 'prompt': f'Warmup request {i}', 'max_tokens': 20})

# Same test as Experiment 2 (fair comparison)
kvq_results = benchmark_concurrent(PROMPTS, max_tokens=N_TOKENS, n_workers=N_REQUESTS, label='KV quant (10 users)')

# Stress test: 20 concurrent (proves capacity gain)
print('\nStress test: 20 concurrent users...')
stress_prompts = PROMPTS * 2
stress_results = benchmark_concurrent(stress_prompts, max_tokens=N_TOKENS, n_workers=20, label='KV quant (20 users)')

results['vLLM+KVQuant'] = kvq_results
print(f'\nKV quant sustained 20 concurrent users: {stress_results["throughput_tps"]:.0f} tok/s')
print(f'Throughput at 10 users: {kvq_results["throughput_tps"]:.0f} tok/s (vs default vLLM: {results["vLLM"]["throughput_tps"]:.0f})')


In [ ]:
plot_comparison(results)


## Experiment 5: Speculative Decoding

Stop the server and restart with a draft model:
```bash
python -m vllm.entrypoints.openai.api_server \
    --model mistralai/Mistral-7B-v0.1 \
    --dtype float16 \
    --enable-prefix-caching \
    --kv-cache-dtype fp8 \
    --spec-method draft_model \
    --spec-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
    --spec-tokens 5 \
    --gpu-memory-utilization 0.90 \
    --port 8000
```

A small draft model proposes tokens, the main model verifies in parallel.
This may OOM on smaller GPUs (needs both models loaded).


In [ ]:
# --- Experiment 5: Speculative Decoding ---
try:
    assert check_server()
    for i in range(5):
        _ = requests.post(f'{BASE_URL}/completions',
            json={'model': MODEL, 'prompt': f'Warmup request {i}', 'max_tokens': 20})

    spec_results = benchmark_concurrent(PROMPTS, max_tokens=N_TOKENS, n_workers=N_REQUESTS, label='speculative')
    results['vLLM+Spec'] = spec_results
    print(f"\nDecode speedup vs default vLLM: {spec_results['throughput_tps']/results['vLLM']['throughput_tps']:.2f}x")

except Exception as e:
    print(f'Speculative decoding failed (likely OOM): {e}')
    print('Expected if VRAM insufficient for target + draft model.')
    results['vLLM+Spec'] = None


In [ ]:
# Only plot if speculative succeeded
if results.get('vLLM+Spec') is not None:
    plot_comparison(results)
else:
    plot_comparison({k: v for k, v in results.items() if v is not None})


## Final Comparison: All Optimizations

Same model. Same GPU. Same prompts. Each flag addresses a different bottleneck:
- **vLLM default:** throughput (PagedAttention + batching)
- **Prefix caching:** TTFT for repeat users
- **KV quantization:** memory capacity (more users)
- **Speculative decoding:** decode speed (ITL)


In [ ]:
# --- Final Comparison ---
valid = {k: v for k, v in results.items() if v is not None}
names = list(valid.keys())
throughputs = [valid[k]['throughput_tps'] for k in names]
ttfts = [valid[k]['avg_ttft'] * 1000 for k in names]
colors = ['#64748b', '#2563eb', '#059669', '#d97706', '#7c3aed'][:len(names)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: throughput
bars = ax1.bar(names, throughputs, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_ylabel('Throughput (tok/s)')
ax1.set_title('Throughput Comparison', fontweight='bold')
ax1.set_xticklabels(names, rotation=15, ha='right', fontsize=9)
for bar, val in zip(bars, throughputs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.0f}',
             ha='center', va='bottom', fontsize=9, fontweight='bold')

# Right: TTFT
ax2.bar(names, ttfts, color=colors, edgecolor='black', linewidth=0.8, alpha=0.7)
ax2.plot(names, ttfts, 'ko-', markersize=8, linewidth=2)
ax2.set_ylabel('Avg TTFT (ms)')
ax2.set_title('Time to First Token', fontweight='bold')
ax2.set_xticklabels(names, rotation=15, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig('demo_f_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
hf_tps = results['HF Baseline']['throughput_tps']
print('\n' + '='*55)
print(f'{"Config":<20} {"tok/s":>8} {"TTFT ms":>8} {"vs HF":>8}')
print('-'*55)
for name, r in valid.items():
    print(f"{name:<20} {r['throughput_tps']:>8.1f} {r['avg_ttft']*1000:>8.0f} {r['throughput_tps']/hf_tps:>7.1f}x")
print('='*55)


## Summary: vLLM Optimization Flags

| Flag | What It Does | Best For |
|------|-------------|----------|
| (default) | PagedAttention + continuous batching | General serving |
| `--enable-prefix-caching` | Reuses KV for shared prefixes | Chat, RAG |
| `--kv-cache-dtype fp8` | Quantizes KV to 8-bit | High concurrency |
| `--speculative-model X` | Draft proposes, main verifies | Low-latency decode |

**Next:** Demo G compares SGLang's RadixAttention vs vLLM's hash-based prefix matching.
